# SETUP

## Imports y rutas

In [ ]:
import os

RUTA_DATOS   = '/kaggle/input/competitions/rsna-pneumonia-detection-challenge'
RUTA_OUTPUTS = '/kaggle/working'

In [ ]:
!pip install pydicom --quiet
!pip install kaggle --upgrade --quiet

## Instalar librerías

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from tqdm.notebook import tqdm
import cv2
import shutil
import subprocess
import json

## Verificar estructura
compruebo qué archivos y carpetas trae el dataset antes de explorar

In [ ]:
for elemento in sorted(os.listdir(RUTA_DATOS)):
    ruta = os.path.join(RUTA_DATOS, elemento)
    if os.path.isdir(ruta):
        n = len(os.listdir(ruta))
        print(f'[dir]  {elemento}/ — {n} archivos')
    else:
        print(f'[file] {elemento}')

##  Leer CSV de labels

In [ ]:
df_labels = pd.read_csv(f'{RUTA_DATOS}/stage_2_train_labels.csv')
print(df_labels.shape)
df_labels.head(10)

# EXPLORACIÓN DEL DATASET

## Distribución de clases

In [ ]:
# verifico el balance entre positivos y negativos porque afecta directamente a la estrategia de entrenamiento
conteo = df_labels.groupby('Target')['patientId'].nunique()
print(conteo)
print(f'\nratio positivos: {conteo[1] / conteo.sum():.2%}')

## Clases detalladas

No Lung Opacity / Not Normal: Sin opacidad pulmonar / No normal (significa que la radiografía muestra alguna anomalía, pero no es la opacidad típica de una neumonía; podría ser otra condición médica).

Lung Opacity: Opacidad pulmonar (este es el grupo positivo para neumonía, donde se ven las manchas blanquecinas en los pulmones).

Normal: Normal (pulmones completamente sanos y sin hallazgos extraños).

In [ ]:
# cargo el csv con las 3 clases para entender cuántos casos son anómalos pero sin neumonía
df_clases = pd.read_csv(f'{RUTA_DATOS}/stage_2_detailed_class_info.csv')
print(df_clases['class'].value_counts())

## Ejemplo DICOM

In [ ]:
# leo un dicom de ejemplo para entender su estructura antes de planificar la conversión a PNG
ruta_train = f'{RUTA_DATOS}/stage_2_train_images'
ejemplo_id  = df_labels['patientId'].iloc[0]
dcm         = pydicom.dcmread(f'{ruta_train}/{ejemplo_id}.dcm')

print(f'shape:             {dcm.pixel_array.shape}')
print(f'dtype:             {dcm.pixel_array.dtype}')
print(f'min/max:           {dcm.pixel_array.min()} / {dcm.pixel_array.max()}')
print(f'PhotometricInterp: {dcm.PhotometricInterpretation}')

Esto describe las propiedades técnicas de una de las imágenes radiográficas (en formato DICOM)

MONOCHROME2 $\rightarrow$ Interpretación fotométrica: Esto es clave en medicina. Significa que el 0 es negro y el valor más alto es blanco.Nota: Si dijera MONOCHROME1, la imagen estaría invertida (el 0 sería blanco y el 255 negro, como un negativo fotográfico).

Al ser MONOCHROME2, puedes visualizarla directamente de forma normal: los huesos y zonas densas se verán blancos/claros, y el aire (los pulmones sanos) se verá negro/oscuro.

## Visualizar ejemplos con bounding box

In [ ]:
# muestro casos positivos y negativos para entender visualmente cómo se ve la neumonía en la radiografía
def mostrar_caso(patient_id, ax):
    dcm   = pydicom.dcmread(f'{ruta_train}/{patient_id}.dcm')
    img   = dcm.pixel_array
    filas = df_labels[df_labels['patientId'] == patient_id]
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{patient_id[:8]}\nTarget={filas["Target"].iloc[0]}')
    for _, fila in filas.iterrows():
        if fila['Target'] == 1:
            rect = patches.Rectangle(
                (fila['x'], fila['y']), fila['width'], fila['height'],
                linewidth=2, edgecolor='red', facecolor='none'
            )
            ax.add_patch(rect)

positivos = df_labels[df_labels['Target'] == 1]['patientId'].unique()[:3]
negativos = df_labels[df_labels['Target'] == 0]['patientId'].unique()[:3]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, pid in zip(axes[0], positivos):
    mostrar_caso(pid, ax)
for ax, pid in zip(axes[1], negativos):
    mostrar_caso(pid, ax)
plt.tight_layout()
plt.show()

Estás viendo un análisis exploratorio de datos (EDA) donde comparan ejemplos positivos de neumonía frente a negativos. Vamos a desglosarlo en dos partes:

1. La fila de arriba: Target = 1 (Casos Positivos)
Estas tres radiografías pertenecen a pacientes que sí tienen neumonía (opacidad pulmonar).

Las cajas rojas (Bounding Boxes): Son las etiquetas (ground truth) que los radiólogos expertos dibujaron a mano. Le indican a tu modelo la ubicación exacta de la infección.

Fíjate que la neumonía suele aparecer en ambos pulmones o en zonas específicas, viéndose como manchas blancas difusas (opacidades) donde normalmente debería verse oscuro (aire).

El objetivo de tu IA: Cuando le pases una imagen nueva, tu modelo tendrá que ser capaz de adivinar y dibujar esas mismas cajas rojas en el lugar correcto.

2. La fila de abajo: Target = 0 (Casos Negativos)
Estas tres radiografías pertenecen a pacientes que no tienen opacidad por neumonía.

Sin cajas rojas: Como el objetivo es detectar opacidad pulmonar por neumonía, aquí no hay ninguna caja que dibujar.

El truco médico (¡Ojo con la tercera imagen!): Mira la imagen de abajo a la derecha (00322d4d). Tiene unas manchas blancas gigantescas y muy marcadas en el pulmón derecho (a tu izquierda). Cualquiera pensaría que está enfermo, ¡y lo está! Pero el médico determinó que no es la opacidad típica de una neumonía (podría ser un tumor, líquido u otra patología).

Esto demuestra por qué el reto es difícil: tu modelo tiene que aprender a diferenciar la opacidad de la neumonía de otras manchas o problemas pulmonares.

¿Qué te dice esto para tu código?
Que estás ante un problema clásico de Detección de Objetos. Tu modelo no solo tendrá que decir "sí o no" (Target 1 o 0), sino que si dice "sí", tendrá que dar las coordenadas de esas cajitas rojas.


## Análisis de bounding boxes

In [ ]:
# analizo tamaños y distribución de las boxes para entender qué escala tienen los focos de neumonía
df_pos = df_labels[df_labels['Target'] == 1].copy()

print(f'total bounding boxes: {len(df_pos)}')
print(f'pacientes con 1 box:  {(df_pos.groupby("patientId").size() == 1).sum()}')
print(f'pacientes con 2+ box: {(df_pos.groupby("patientId").size() > 1).sum()}')
print()
print(df_pos[['x', 'y', 'width', 'height']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_pos['width'],  bins=40, color='steelblue')
axes[0].set_title('distribución width')
axes[1].hist(df_pos['height'], bins=40, color='steelblue')
axes[1].set_title('distribución height')
plt.tight_layout()
plt.show()

In [ ]:
!git clone --branch fran https://github.com/franciscofdzfer/RSNAPneumonia

In [ ]:
pacientes_multi = df_pos.groupby('patientId').size()
paciente_multi  = pacientes_multi[pacientes_multi > 1].index[0]

print(df_labels[df_labels['patientId'] == paciente_multi])

# PREPROCESADO

##  Función de conversión DICOM → PNG

In [ ]:

# defino la función de conversión antes del bucle para poder testearla sobre pocas imágenes primero
def dicom_a_png(ruta_dcm, ruta_png, tamanio=512):
    dcm = pydicom.dcmread(ruta_dcm)
    img = dcm.pixel_array.astype(np.float32)

    # invierto si la imagen está en MONOCHROME1 (blanco=aire, negro=tejido)
    if dcm.PhotometricInterpretation == 'MONOCHROME1':
        img = img.max() - img

    # normalizo a 0-255
    img = (img - img.min()) / (img.max() - img.min() + 1e-6) * 255
    img = img.astype(np.uint8)

    # convierto a 3 canales para compatibilidad con modelos ImageNet
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img = cv2.resize(img, (tamanio, tamanio))

    cv2.imwrite(ruta_png, img)

## Test de conversión sobre 5 imágenes

In [ ]:

# pruebo la función sobre 5 imágenes antes de lanzar el bucle completo con las 26k
ruta_png_train = os.path.join(RUTA_OUTPUTS, 'png_train')
os.makedirs(ruta_png_train, exist_ok=True)

ids_test = df_labels['patientId'].unique()[:5]

for pid in ids_test:
    ruta_dcm = f'{ruta_train}/{pid}.dcm'
    ruta_png = f'{ruta_png_train}/{pid}.png'
    dicom_a_png(ruta_dcm, ruta_png)
    print(f'convertido: {pid}.png — existe: {os.path.exists(ruta_png)}')

## Verificar visualmente los PNGs convertidos

In [ ]:
# compruebo que la conversión mantiene la calidad visual y las proporciones correctas
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, pid in zip(axes, ids_test):
    ruta_png = f'{ruta_png_train}/{pid}.png'
    img      = cv2.imread(ruta_png)
    img      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(pid[:8])
    ax.axis('off')

plt.tight_layout()
plt.show()

## Bucle completo de conversión train

In [ ]:

# convierte las  imágenes de train completas
ids_train = df_labels['patientId'].unique()
errores   = []

for pid in tqdm(ids_train, desc='convirtiendo train'):
    ruta_dcm = f'{ruta_train}/{pid}.dcm'
    ruta_png = f'{ruta_png_train}/{pid}.png'
    if os.path.exists(ruta_png):
        continue
    try:
        dicom_a_png(ruta_dcm, ruta_png)
    except Exception as e:
        errores.append((pid, str(e)))

print(f'\nconvertidos: {len(os.listdir(ruta_png_train))}')
print(f'errores:     {len(errores)}')

## Bucle completo de conversión test

In [ ]:
ruta_test       = f'{RUTA_DATOS}/stage_2_test_images'
ruta_png_test   = os.path.join(RUTA_OUTPUTS, 'png_test')
os.makedirs(ruta_png_test, exist_ok=True)

ids_test_full = [f.replace('.dcm', '') for f in os.listdir(ruta_test)]
errores_test  = []

for pid in tqdm(ids_test_full, desc='convirtiendo test'):
    ruta_dcm = f'{ruta_test}/{pid}.dcm'
    ruta_png = f'{ruta_png_test}/{pid}.png'
    if os.path.exists(ruta_png):
        continue
    try:
        dicom_a_png(ruta_dcm, ruta_png)
    except Exception as e:
        errores_test.append((pid, str(e)))

print(f'\nconvertidos: {len(os.listdir(ruta_png_test))}')
print(f'errores:     {len(errores_test)}')

## comprimir

In [ ]:
# comprimo train
shutil.make_archive(
    f'{RUTA_OUTPUTS}/png_train',
    'zip',
    RUTA_OUTPUTS,
    'png_train'
)

print('zip creado')

In [ ]:
# comprimo test
shutil.make_archive(
    f'{RUTA_OUTPUTS}/png_test',
    'zip',
    RUTA_OUTPUTS,
    'png_test'
)

print('zip test creado')

## subir a Kaggle

In [ ]:
ruta_upload = f'{RUTA_OUTPUTS}/upload_train'
os.makedirs(ruta_upload, exist_ok=True)

shutil.copy(f'{RUTA_OUTPUTS}/png_train.zip', ruta_upload)

metadata = {
    "title": "rsna-png-512",
    "id": "franciscofdzfer/rsna-png-512",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(f'{ruta_upload}/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

print('listo')

In [ ]:
subprocess.run([
    'kaggle', 'datasets', 'create',
    '-p', ruta_upload,
    '--dir-mode', 'zip'
])

In [ ]:
ruta_upload_test = f'{RUTA_OUTPUTS}/upload_test'
os.makedirs(ruta_upload_test, exist_ok=True)

shutil.copy(f'{RUTA_OUTPUTS}/png_test.zip', ruta_upload_test)

metadata_test = {
    "title": "rsna-png-512-test",
    "id": "franciscofdzfer/rsna-png-512-test",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(f'{ruta_upload_test}/dataset-metadata.json', 'w') as f:
    json.dump(metadata_test, f)

subprocess.run([
    'kaggle', 'datasets', 'create',
    '-p', ruta_upload_test,
    '--dir-mode', 'zip'
])

In [ ]:
subprocess.run(['kaggle', 'datasets', 'list', '--user', 'franciscofdzfer'])